# Student Distillation: German Customer Support (QLoRA)

Trains `Qwen2.5-1.5B-Instruct` on teacher-generated (prompt, response) pairs via **sequence-level knowledge distillation**.   
The student learns the teacher's domain style and response format without ever seeing the teacher's internal probability distributions.   

**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → `[03 Student Distillation]` → 04 Evaluation   

**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).   
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/03-student-distillation)

## 1. Setup

Install dependencies and check GPU.  
Local users: skip the pip cell — install via `pip install -r requirements.txt` instead.  
Checks if CUDA capable GPU is available

In [1]:
%%capture
!pip install -q --upgrade unsloth trl datasets transformers peft bitsandbytes accelerate huggingface_hub python-dotenv

In [2]:
!nvidia-smi
import torch
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Fri May 15 22:12:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Authenticate

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.   
**Local:** create a `.env` file with `HF_TOKEN=your_token`.   

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env locally, no-op on Kaggle

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except ImportError:
    print("kaggle_secrets not available — using .env fallback.")
except Exception as e:
    print(f"Kaggle Secrets found but could not load HF_TOKEN: {e}")

HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    print(f"HF_TOKEN set. ({HF_TOKEN[:4]}...{HF_TOKEN[-4:]})")
else:
    print("WARNING: HF_TOKEN is not set. The save cell will fail.")

HF_TOKEN loaded from Kaggle Secrets.
HF_TOKEN set. (hf_R...nFKm)


## 3. Load Student Base Model

**Why `Qwen2.5-1.5B-Instruct`?**
- Half the size of the teacher (1.5B vs 3B) — the point of distillation is a smaller deployable model
- Same model family as the teacher: shares tokenizer, chat template, and pretraining data, so style transfer is efficient
- Fits in ~1.5 GB VRAM in 4-bit — leaves plenty of headroom for training on a T4   
- Apache 2.0 license   

In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None           # auto-detect: float16 on T4, bfloat16 on Ampere+
LOAD_IN_4BIT = True    # QLoRA: quantize base weights to 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Loaded: {model.config._name_or_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit


## 4. Apply QLoRA Adapter

Same LoRA configuration as the teacher (Notebook 01) — `r=16`, all attention and MLP projections targeted.   
The adapter is initialised from scratch; the base weights remain frozen.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable parameters: 18,464,768 / 907,081,216 (2.04%)


## 5. Load Distillation Dataset

The dataset is the 232 (prompt, response) pairs generated by the teacher in Notebook 02.   
It is stored alongside the LoRA adapter on Hugging Face Hub and downloaded directly from there — no local file needed.   

In [6]:
import json
from huggingface_hub import hf_hub_download

DATA_REPO = "Feyerade/german-support-qwen-lora-adapter"

file_path = hf_hub_download(
    repo_id=DATA_REPO,
    filename="teacher_generated_data.json",
    token=HF_TOKEN,
)

with open(file_path, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} teacher-generated examples")
print(f"\nExample entry:")
print(f"  instruction: {raw_data[0]['instruction'][:150]}")
print(f"  output:      {raw_data[0]['output'][:300]}...")

Loaded 232 teacher-generated examples

Example entry:
  instruction: Meine Lieferung sollte gestern ankommen, aber im Tracking steht immer noch 'in Bearbeitung'.
  output:      Das ist aergerlich! Eine verdaechtige Verzuerkung des Drittlersystems:

1. Kontaktieren Sie den Drittler (z.B. Post oder Deutsche E-Handel)
2. Fragen Sie nach dem Status Ihrer Lieferung
3. Falls verloren - wir klaeren das sofort bei dem Drittler

Bitte versuchen Sie auch, Ihre E-Mail-Adresse bei uns...


## 6. Format Dataset

The goal of this project is style distillation: teaching the student to respond in the same structured, professional format as the teacher — numbered steps, polite phrasing, concise answers. That format is the main thing being transferred here, not factual knowledge.

For that transfer to work, the training data must use the exact same prompt structure the student will use when generating responses. Qwen2.5 uses a chat format with clearly labelled roles (`system`, `user`, `assistant`). `apply_chat_template`handles the formatting automatically:

    <|im_start|>system
    Du bist ein professioneller Kundenservice-Mitarbeiter ...
    <|im_start|>user
    Mein Paket ist nicht angekommen.
    <|im_start|>assistant
    Das verstehe ich — lassen Sie mich das prüfen ...

The system prompt is identical to the one used in Notebook 02 when the teacher generated the responses. This is important: the student is trained on data produced under the exact same instructions it will operate under — so the response style it learns is directly the style it needs to reproduce.

In [7]:
import random
from datasets import Dataset

SYSTEM_PROMPT = (
    "Du bist ein professioneller Kundenservice-Mitarbeiter. "
    "Antworte freundlich, loesungsorientiert und auf Deutsch. "
    "Halte deine Antworten unter 150 Woertern."
)

def format_example(example, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted = [{"text": format_example(ex, tokenizer)} for ex in raw_data]
dataset = Dataset.from_list(formatted)

idx = random.randint(0, len(dataset) - 1)
print(f"Sample (index {idx}):\n")
print(dataset[idx]["text"])

Sample (index 163):

<|im_start|>system
Du bist ein professioneller Kundenservice-Mitarbeiter. Antworte freundlich, loesungsorientiert und auf Deutsch. Halte deine Antworten unter 150 Woertern.<|im_end|>
<|im_start|>user
Ich bekomme keine Updates zum Versand.<|im_end|>
<|im_start|>assistant
Das tut mir leid! Hier sind die Schritte zurueckzuleiten:

1. Pruefen Sie continuous Updates auf Ihrer Karte
2. Gehen Sie zu 'Einstellungen' > 'Apps' > 'Yandex Paketverdienst'
3. Klicken Sie auf 'Sofort aktualisieren'

Falls das Problem besteht, senden wir Ihnen eine E-Mail ganz persoenlich mit einem Shortcut zur App-Aktualisierung.<|im_end|>



## 7. Train

Standard SFT on (prompt, teacher-response) pairs — no access to teacher logits required.   
This is **sequence-level knowledge distillation** (Kim & Rush, 2016): the student mimics the teacher's outputs rather than its internal probability distributions.   

Key hyperparameters:
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=4` → effective batch size of 8   
- `max_steps=200` — ~6 epochs over 232 examples; loss convergence verified empirically   
- `optim="adamw_8bit"` — memory-efficient optimizer from bitsandbytes
- `report_to="none"` — disables W&B/HF logging

In [8]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=200,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

print(f"\nSteps:         {trainer_stats.global_step}")
print(f"Average training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime:       {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:   {trainer_stats.metrics['train_samples_per_second']:.2f}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/232 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 232 | Num Epochs = 7 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.462868
20,1.407182
30,1.136864
40,0.993267
50,0.933357
60,0.866754
70,0.773343
80,0.733412
90,0.657262
100,0.551458


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.



Steps:         200
Average training loss: 0.7028
Runtime:       343s
Samples/sec:   4.67


## 8. Qualitative Evaluation

Sanity check on the same held-out queries used in Notebook 01.   
This lets you compare student and teacher outputs side-by-side without running Notebook 04.   
Systematic benchmarking (BERTScore, latency, VRAM) is in Notebook 04.   

In [9]:
import transformers
import warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)

test_queries = [
    "Mein Paket ist seit 2 Wochen nicht angekommen.",
    "Kann ich auch per Rechnung bezahlen?",
    "Ich moechte meine Adresse aendern.",
    "Das Produkt entspricht nicht der Beschreibung.",
    "Gibt es das auch in Groesse XL?",
]

for query in test_queries:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    print("=" * 60)
    print(f"Query:    {query}")
    print("=" * 60)
    print(f"Response: {response}")

Query:    Mein Paket ist seit 2 Wochen nicht angekommen.
Response: Das tut mir leid! Das kann verschiedene Gruende haben:

1. Pruefen Sie Ihre Posteingangsbox: Manchmal kommen Pakete umgehend vor.
2. Kontaktieren Sie die Sendeanbieter (z.B. DHL, UPS): Sie muessen sicher sein, dass das Paket versandt wurde.
3. Anformen Sie den Verfolgen-Link in unserem System: Wir pruefen ob das Paket verloren gegangen ist.

Bitte kontaktieren Sie uns so schnell wie moeglich — wir klaeren das sofort!
Query:    Kann ich auch per Rechnung bezahlen?
Response: Natürlich! Rechnungen sind gerne benoetigt. Wir klaeren das sofort:
Query:    Ich moechte meine Adresse aendern.
Response: Kein Problem! Die Adressaenderung ist einfach: 

1. Loggen Sie sich ein und gehen Sie zu 'Einstellungen' > 'Adresse'
2. Bearbeiten oder loeschen Sie Ihre alte Adresse voruebergehend
3. Speichern Sie die neue Adresse

Falls Sie Helpen wollen, steuere ich Sie durch.
Query:    Das Produkt entspricht nicht der Beschreibung.
Response: 

## 9. Push to Hugging Face Hub

Saves the merged student model (base + LoRA weights) and pushes to Hugging Face Hub.   
Switch `if False` to `if True` to activate a section.   

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.   
**Local:** set `HF_TOKEN` as an environment variable.   

The merged model is pushed as a standalone checkpoint — no adapter files needed at inference time.   

In [10]:
REPO_ID = "Feyerade/german-support-student-1.5b-distilled"

# Save merged model locally
if False:
    model.save_pretrained_merged(
        save_directory="german_support_student_merged",
        tokenizer=tokenizer,
        save_method="merged_16bit",
    )
    print("Merged student model saved locally!")

# Push merged model directly to Hugging Face Hub
if False:
    model.push_to_hub_merged(
        REPO_ID,
        tokenizer=tokenizer,
        save_method="merged_16bit",
        token=HF_TOKEN,
    )
    print(f"Pushed to: https://huggingface.co/{REPO_ID}")